# Differential Equations — Session 34
## Section 7.5: The Dirac Delta Function

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to distinguish a short pulse from an ideal impulse; describe the delta distribution through its integral and sampling properties; compute its Laplace transform; solve impulse-forced IVPs; derive the jump in velocity or momentum; and compare narrow finite pulses with the idealized delta response.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Rectangular unit impulses |
| 18–35 min | Delta distribution and sampling |
| 35–52 min | Laplace transform of a delayed delta |
| 52–78 min | Impulse-forced oscillator |
| 78–88 min | Narrow-pulse approximation |
| 88–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import quad, solve_ivp
from scipy.signal import fftconvolve
from scipy.linalg import expm, eig
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def unit_step(t, a=0.0):
    t = np.asarray(t)
    return (t >= a).astype(float)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 7.5-A — Rectangular unit impulse

For width $a>0$ centered near $t_0$, a rectangular approximation may be chosen with height $1/a$ and total area $1$.

### Definition 7.5-B — Dirac delta distribution

The delta is characterized formally by

$$
\delta(t-t_0)=0
\quad\text{for }t\ne t_0,
$$

and

$$
\int_{-\infty}^{\infty}\delta(t-t_0)\,dt=1.
$$

It is not an ordinary function.

### Theorem 7.5-C — Sampling property

For continuous $f$,

$$
\int_{-\infty}^{\infty}
f(t)\delta(t-t_0)\,dt
=
f(t_0).
$$

### Theorem 7.5-D — Laplace transform

For $t_0>0$,

$$
\mathcal L\{\delta(t-t_0)\}
=
e^{-st_0}.
$$

### Proposition 7.5-E — State jump under impulse forcing

For

$$
m y''+c y'+k y=J\delta(t-t_0),
$$

integrating across $t_0$ gives

$$
m\big[y'(t_0^+)-y'(t_0^-)\big]=J.
$$

The displacement remains continuous, while velocity jumps by $J/m$.

### Classroom Checkpoint — Impulse Jump

For

$$
my''+cy'+ky=J\delta(t-t_0),
$$

what jump occurs across the impulse?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Narrow pulses with fixed area

In [ ]:
t = np.linspace(0, 10, 3000)
t0 = 5

for width in [2, 1, 0.5, 0.2]:
    pulse = np.where(np.abs(t-t0) <= width/2, 1/width, 0)
    plt.plot(t, pulse, label=f"width={width}")

plt.xlabel("t")
plt.ylabel("pulse height")
plt.title("Unit-area rectangular impulses")
plt.legend()
plt.show()

In [ ]:
def impulse_approximation(width=0.5, t0=5.0):
    t = np.linspace(0, 10, 5000)
    pulse = np.where(np.abs(t-t0) <= width/2, 1/width, 0)
    area = np.trapz(pulse, t)
    plt.plot(t, pulse)
    plt.axvline(t0, linestyle="--")
    plt.xlabel("t")
    plt.ylabel("amplitude")
    plt.title(fr"Width={width:.3f}, numerical area={area:.5f}")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        impulse_approximation,
        width=FloatSlider(min=0.05, max=2, step=0.05, value=0.5),
        t0=FloatSlider(min=1, max=9, step=0.25, value=5)
    )
else:
    impulse_approximation()

## 2. Sampling property

A narrow unit-area pulse samples a continuous function near its center.

In [ ]:
def sampling_experiment(width=0.4, t0=3.0):
    t = np.linspace(0, 8, 10000)
    f = np.exp(-0.2*t)*np.cos(2*t)
    pulse = np.where(np.abs(t-t0) <= width/2, 1/width, 0)
    sampled = np.trapz(f*pulse, t)

    plt.plot(t, f, label="f(t)")
    plt.plot(t, pulse, label="impulse approximation")
    plt.axvline(t0, linestyle="--")
    plt.legend()
    plt.show()

    print("weighted integral:", sampled)
    print("target f(t0):", np.exp(-0.2*t0)*np.cos(2*t0))

if WIDGETS_AVAILABLE:
    interact(
        sampling_experiment,
        width=FloatSlider(min=0.05, max=1.5, step=0.05, value=0.4),
        t0=FloatSlider(min=0.5, max=7, step=0.25, value=3)
    )
else:
    sampling_experiment()

## 3. Transform of a delayed impulse

The sampling property gives

$$
\int_0^\infty e^{-st}\delta(t-t_0)\,dt
=
e^{-st_0}.
$$

Thus the delay factor $e^{-st_0}$ has two interpretations:

- delayed ordinary forcing through the second translation theorem;
- an ideal impulse located at $t_0$.

## 4. Impulse-forced oscillator

Consider

$$
y''+\omega^2y=J\delta(t-t_0),
\qquad
y(0)=0,
\qquad
y'(0)=0.
$$

The transformed solution is

$$
Y(s)
=
\frac{Je^{-st_0}}{s^2+\omega^2}.
$$

Therefore

$$
y(t)
=
\frac{J}{\omega}
u(t-t_0)
\sin\big(\omega(t-t_0)\big).
$$

In [ ]:
def ideal_impulse_response(omega=2.0, J=1.0, t0=4.0):
    t = np.linspace(0, 15, 1200)
    y = J/omega*unit_step(t, t0)*np.sin(omega*(t-t0))
    velocity = J*unit_step(t, t0)*np.cos(omega*(t-t0))

    plt.plot(t, y, label="displacement")
    plt.plot(t, velocity, label="velocity")
    plt.axvline(t0, linestyle="--", label="impulse time")
    plt.legend()
    plt.title("Ideal impulse response")
    plt.show()

    print("predicted velocity jump:", J)

if WIDGETS_AVAILABLE:
    interact(
        ideal_impulse_response,
        omega=FloatSlider(min=0.5, max=6, step=0.25, value=2),
        J=FloatSlider(min=-3, max=3, step=0.25, value=1),
        t0=FloatSlider(min=0, max=10, step=0.25, value=4)
    )
else:
    ideal_impulse_response()

## 5. Compare a narrow pulse with the ideal response

Replace $\delta(t-t_0)$ by a rectangular pulse of width $a$ and height $1/a$.

In [ ]:
def finite_vs_ideal(width=0.4, omega=2.0, t0=4.0):
    t = np.linspace(0, 15, 3000)
    start, stop = t0-width/2, t0+width/2

    def rhs(t, z):
        forcing = 1/width if start <= t <= stop else 0.0
        return [z[1], forcing-omega**2*z[0]]

    sol = solve_ivp(rhs, (0, 15), [0, 0], t_eval=t,
                    rtol=1e-9, atol=1e-11, max_step=width/20)

    ideal = 1/omega*unit_step(t, t0)*np.sin(omega*(t-t0))

    plt.plot(t, sol.y[0], label="finite pulse")
    plt.plot(t, ideal, linestyle="--", label="ideal delta")
    plt.axvline(t0, linestyle=":")
    plt.legend()
    plt.title("Narrow pulse converges to ideal impulse response")
    plt.show()

    print("maximum displayed difference:", np.max(np.abs(sol.y[0]-ideal)))

if WIDGETS_AVAILABLE:
    interact(
        finite_vs_ideal,
        width=FloatSlider(min=0.05, max=1.5, step=0.05, value=0.4),
        omega=FloatSlider(min=0.5, max=5, step=0.25, value=2),
        t0=FloatSlider(min=1, max=8, step=0.25, value=4)
    )
else:
    finite_vs_ideal()

## 6. Existing motion before the impulse

If the oscillator is already moving, the impulse adds a shifted impulse response to the preexisting homogeneous motion.

In [ ]:
t = np.linspace(0, 15, 1200)
omega, t0, J = 1.5, 5.0, 1.2
baseline = np.cos(omega*t)
added = J/omega*unit_step(t, t0)*np.sin(omega*(t-t0))
total = baseline+added

plt.plot(t, baseline, label="motion without impulse")
plt.plot(t, total, label="motion with impulse")
plt.axvline(t0, linestyle="--")
plt.legend()
plt.title("Impulse modifies amplitude and phase")
plt.show()

## Classroom Checkpoint — Exit Check

For

$$
2y''+8y=\delta(t-3),
\qquad
y(0)=y'(0)=0,
$$

find the solution.

> Pause here. Let students commit to an answer before running the next cell.